In [8]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import random
import os
from collections import Counter
from nltk.tokenize import word_tokenize
import time

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU: True
Tesla T4


In [9]:
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [10]:
from gensim.models import KeyedVectors

W2V_PATH = "/kaggle/input/datasets/sugataghosh/google-word2vec/GoogleNews-vectors-negative300.bin"

print("Loading word2vec...")
t0 = time.time()
word2vec = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True)
print(f"Done: {len(word2vec)} vectors, dim={word2vec.vector_size}, {time.time()-t0:.1f}s")

Loading word2vec...
Done: 3000000 vectors, dim=300, 22.6s


In [14]:
from nltk.tree import Tree
import re

TREE_DIR = "/kaggle/input/datasets/chiranjivipanda13/stanford-phrase-level/trees"

def clean_tokens(leaves):
    s = " ".join(leaves)
    s = s.replace("-LRB-", "(").replace("-RRB-", ")")
    s = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", s)
    s = re.sub(r"\s{2,}", " ", s)
    return s.strip().lower().split()

def label_to_binary(label):
    if label <= 1:
        return 0
    if label >= 3:
        return 1
    return None

def load_trees(path, phrases):
    texts, labels, seen = [], [], set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tree = Tree.fromstring(line)
            nodes = tree.subtrees() if phrases else [tree]
            for node in nodes:
                b = label_to_binary(int(node.label()))
                if b is None:
                    continue
                toks = clean_tokens(node.leaves())
                if not toks:
                    continue
                if phrases:
                    key = " ".join(toks)
                    if key in seen:
                        continue
                    seen.add(key)
                texts.append(toks)
                labels.append(b)
    return texts, labels

train_texts, train_labels = load_trees(os.path.join(TREE_DIR, "train.txt"), phrases=True)
dev_texts,   dev_labels   = load_trees(os.path.join(TREE_DIR, "dev.txt"),   phrases=False)
test_texts,  test_labels  = load_trees(os.path.join(TREE_DIR, "test.txt"),  phrases=False)

print(f"Train phrases : {len(train_texts):,}")
print(f"Dev sentences : {len(dev_texts):,}")
print(f"Test sentences: {len(test_texts):,}")
print(f"Train balance : pos={sum(train_labels)} neg={len(train_labels)-sum(train_labels)}")
print("Example:", train_texts[0], train_labels[0])

Train phrases : 71,507
Dev sentences : 872
Test sentences: 1,821
Train balance : pos=39367 neg=32140
Example: ['the', 'rock', 'is', 'destined', 'to', 'be', 'the', '21st', 'century', "'s", 'new', '``', 'conan', "''", 'and', 'that', 'he', "'s", 'going', 'to', 'make', 'a', 'splash', 'even', 'greater', 'than', 'arnold', 'schwarzenegger', ',', 'jean', 'claud', 'van', 'damme', 'or', 'steven', 'segal'] 1


In [15]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word_counts = Counter(w for s in train_texts for w in s)
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for w, _ in word_counts.most_common():
    vocab[w] = len(vocab)

vocab_size = len(vocab)
embed_dim = 300
print("Vocab size:", vocab_size)

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
found_vecs = []
found, oov = 0, 0

for w, i in vocab.items():
    if w in (PAD_TOKEN, UNK_TOKEN):
        continue
    if w in word2vec:
        v = word2vec[w]
        embedding_matrix[i] = v
        found_vecs.append(v)
        found += 1
    else:
        oov += 1

var = np.var(np.array(found_vecs))
a = np.sqrt(3.0 * var)
print(f"Found in w2v: {found} | OOV: {oov} | OOV init U[-{a:.4f},{a:.4f}]")

rng = np.random.RandomState(SEED)
for w, i in vocab.items():
    if w == PAD_TOKEN:
        continue
    if w == UNK_TOKEN or w not in word2vec:
        embedding_matrix[i] = rng.uniform(-a, a, embed_dim)

pretrained_weights = torch.FloatTensor(embedding_matrix)

import gc
del word2vec, found_vecs
gc.collect()

Vocab size: 15146
Found in w2v: 13910 | OOV: 1234 | OOV init U[-0.3059,0.3059]


60

In [16]:
max_len = max(len(t) for t in train_texts + dev_texts + test_texts)
print("Max len:", max_len)

class SSTDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        toks = self.texts[idx]
        ids = [self.vocab.get(w, self.vocab[UNK_TOKEN]) for w in toks]
        if len(ids) < self.max_len:
            ids = ids + [0] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return torch.LongTensor(ids), torch.LongTensor([self.labels[idx]])

BATCH_SIZE = 50
train_loader = DataLoader(SSTDataset(train_texts, train_labels, vocab, max_len), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SSTDataset(dev_texts,   dev_labels,   vocab, max_len), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SSTDataset(test_texts,  test_labels,  vocab, max_len), batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Dev batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))

Max len: 53
Train batches: 1431
Dev batches:   18
Test batches:  37


In [18]:
class KimCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, num_filters,
                 filter_sizes, dropout, mode, pretrained_weights, pad_idx=0):
        super().__init__()
        self.mode = mode

        if mode == "multichannel":
            self.emb_static = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.emb_finetune = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.emb_static.weight.data.copy_(pretrained_weights)
            self.emb_finetune.weight.data.copy_(pretrained_weights)
            self.emb_static.weight.requires_grad = False
            in_channels = 2
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            if mode in ("static", "non-static"):
                self.embedding.weight.data.copy_(pretrained_weights)
            if mode == "static":
                self.embedding.weight.requires_grad = False
            in_channels = 1

        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels, num_filters, (h, embed_dim))
            for h in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        if self.mode == "multichannel":
            e1 = self.emb_static(x)
            e2 = self.emb_finetune(x)
            x = torch.stack([e1, e2], dim=1)
        else:
            x = self.embedding(x).unsqueeze(1)

        pooled = []
        for conv in self.convs:
            c = F.relu(conv(x)).squeeze(3)
            c = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled.append(c)

        z = torch.cat(pooled, dim=1)
        z = self.dropout(z)
        return self.fc(z)

    def constrain_fc_norm(self, max_norm=3.0):
        with torch.no_grad():
            norms = self.fc.weight.data.norm(dim=1, keepdim=True)
            desired = torch.clamp(norms, max=max_norm)
            self.fc.weight.data *= desired / (norms + 1e-8)

In [20]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for bx, by in loader:
        bx = bx.to(device); by = by.squeeze(1).to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        model.constrain_fc_norm(3.0)
        total_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(1) == by).sum().item()
        total += bx.size(0)
    return total_loss/total, correct/total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device); by = by.squeeze(1).to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(1) == by).sum().item()
            total += bx.size(0)
    return total_loss/total, correct/total

In [21]:
NUM_FILTERS = 100
FILTER_SIZES = (3, 4, 5)
DROPOUT = 0.5
NUM_CLASSES = 2
NUM_EPOCHS = 25
PATIENCE = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

results = {}

for mode in ["rand", "static", "non-static", "multichannel"]:
    print(f"\n===== CNN-{mode} =====")
    set_seed(SEED)
    model = KimCNN(vocab_size, embed_dim, NUM_CLASSES, NUM_FILTERS,
                   FILTER_SIZES, DROPOUT, mode, pretrained_weights).to(DEVICE)
    optimizer = torch.optim.Adadelta(model.parameters(), rho=0.95)
    criterion = nn.CrossEntropyLoss()

    best_val, best_test, patience_c = 0.0, 0.0, 0
    for epoch in range(NUM_EPOCHS):
        tr_l, tr_a = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        vl_l, vl_a = evaluate(model, val_loader, criterion, DEVICE)
        te_l, te_a = evaluate(model, test_loader, criterion, DEVICE)
        print(f"  ep {epoch+1:2d} | train {tr_a:.4f} | dev {vl_a:.4f} | test {te_a:.4f}")
        if vl_a > best_val:
            best_val, best_test, patience_c = vl_a, te_a, 0
        else:
            patience_c += 1
            if patience_c >= PATIENCE:
                print(f"  early stop at epoch {epoch+1}")
                break

    results[mode] = best_test
    print(f"  BEST test for CNN-{mode}: {best_test:.4f}")

print("\n===== FINAL (SST-2 test accuracy) =====")
paper = {"rand": 82.7, "static": 86.8, "non-static": 87.2, "multichannel": 88.1}
print(f"{'Model':<22}{'Ours':>8}{'Paper':>8}{'Diff':>8}")
for m in ["rand", "static", "non-static", "multichannel"]:
    o = results[m]*100
    p = paper[m]
    print(f"CNN-{m:<18}{o:>7.1f}%{p:>7.1f}%{o-p:>+7.1f}%")

Device: cuda

===== CNN-rand =====
  ep  1 | train 0.6713 | dev 0.7018 | test 0.7177
  ep  2 | train 0.7805 | dev 0.7179 | test 0.7375
  ep  3 | train 0.8238 | dev 0.6984 | test 0.6705
  ep  4 | train 0.8501 | dev 0.7534 | test 0.7562
  ep  5 | train 0.8699 | dev 0.6606 | test 0.6875
  ep  6 | train 0.8827 | dev 0.7477 | test 0.7518
  ep  7 | train 0.8936 | dev 0.7294 | test 0.7474
  ep  8 | train 0.8977 | dev 0.7603 | test 0.7589
  ep  9 | train 0.9073 | dev 0.7580 | test 0.7512
  ep 10 | train 0.9132 | dev 0.6778 | test 0.7013
  ep 11 | train 0.9172 | dev 0.7603 | test 0.7600
  ep 12 | train 0.9232 | dev 0.6548 | test 0.6908
  ep 13 | train 0.9269 | dev 0.7167 | test 0.7326
  early stop at epoch 13
  BEST test for CNN-rand: 0.7589

===== CNN-static =====
  ep  1 | train 0.8415 | dev 0.8028 | test 0.8138
  ep  2 | train 0.8859 | dev 0.8280 | test 0.8248
  ep  3 | train 0.9045 | dev 0.7821 | test 0.7743
  ep  4 | train 0.9170 | dev 0.8314 | test 0.8440
  ep  5 | train 0.9232 | dev 0.83